In [ ]:
# Retry: fix the multi-line string quoting and run again.
from pathlib import Path
import shutil, runpy, pandas as pd
from textwrap import dedent


"""
CAPEC v3.9 extractor -> CSV + JSONL

Fields per attack pattern:
- capec_id
- description
- skilllevel (list of Skill/@Level values)
- skilldescription (list of Skill text, aligned with skilllevel)
- consequenceImpact (list[list[str]] impacts per consequence)
- consequenceScope (list[list[str]] scopes per consequence)
- consequenceNote (list[list[str]] notes per consequence)
- attackStep (list of step numbers/labels)
- attackStepPhase (list aligned with attackStep)
- attackStepDescription (list aligned with attackStep)
- attackStepTechnique (list[list[str]] techniques per step)
- cwe_id (list of related CWE IDs, formatted like 'cwe-120')

Outputs:
- cleaned data/capec_extracted.jsonl
- cleaned data/capec_summary.csv
"""
import json
from pathlib import Path
import xml.etree.ElementTree as ET
import csv

INPUT = Path("data/capec_v3.9.xml")
OUT_DIR = Path("cleaned data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
JSONL_OUT = OUT_DIR / "capec_extracted.jsonl"
CSV_OUT = OUT_DIR / "capec_summary.csv"

def die(msg: str):
    raise SystemExit(msg)

if not INPUT.exists():
    die(f"Input file not found: {INPUT.resolve()} (place capec_v3.9.xml in data/)")

tree = ET.parse(INPUT)
root = tree.getroot()

# namespace handling
ns = {}
if root.tag.startswith("{"):
    uri = root.tag.split("}")[0].strip("{")
    ns['cap'] = uri
else:
    ns['cap'] = ''

def q(tag: str) -> str:
    return f"{{{ns['cap']}}}{tag}" if ns['cap'] else tag

def text_of(elem):
    if elem is None:
        return ""
    return "".join(elem.itertext()).strip()

items = []
for ap in root.findall(".//" + q("Attack_Pattern")):
    ap_id = ap.get("ID") or ""
    capec_id = f"CAPEC-{ap_id}" if ap_id else ""
    description = text_of(ap.find(q("Description")))

    # Skills_Required
    skill_levels, skill_descs = [], []
    skills_parent = ap.find(q("Skills_Required"))
    if skills_parent is not None:
        for skill in skills_parent.findall(q("Skill")):
            lvl = (skill.get("Level") or "").strip()
            skill_levels.append(lvl)
            skill_descs.append(text_of(skill))

    # Consequences
    consequence_impacts, consequence_scopes, consequence_notes = [], [], []
    cp = ap.find(q("Consequences"))
    if cp is not None:
        for cons in cp.findall(q("Consequence")):
            impacts = [text_of(x) for x in cons.findall(q("Impact")) if text_of(x)]
            scopes = [text_of(x) for x in cons.findall(q("Scope")) if text_of(x)]
            notes = [text_of(x) for x in cons.findall(q("Note")) if text_of(x)]
            consequence_impacts.append(impacts)
            consequence_scopes.append(scopes)
            consequence_notes.append(notes)

    # Execution_Flow -> Attack_Step
    attack_steps, attack_phases, attack_descs, attack_techs = [], [], [], []
    ef = ap.find(q("Execution_Flow"))
    if ef is not None:
        for step in ef.findall(q("Attack_Step")):
            attack_steps.append(text_of(step.find(q("Step"))))
            attack_phases.append(text_of(step.find(q("Phase"))))
            attack_descs.append(text_of(step.find(q("Description"))))
            techniques = [text_of(t) for t in step.findall(q("Technique")) if text_of(t)]
            attack_techs.append(techniques)

    # Related_Weaknesses -> CWE_IDs
    cwe_ids = []
    rw = ap.find(q("Related_Weaknesses"))
    if rw is not None:
        for r in rw.findall(q("Related_Weakness")):
            cwe_raw = r.get("CWE_ID") or ""
            if cwe_raw:
                cwe_ids.append(f"cwe-{cwe_raw}".lower())

    item = {
        "capec_id": capec_id,
        "description": description,
        "skilllevel": skill_levels,
        "skilldescription": skill_descs,
        "consequenceImpact": consequence_impacts,
        "consequenceScope": consequence_scopes,
        "consequenceNote": consequence_notes,
        "attackStep": attack_steps,
        "attackStepPhase": attack_phases,
        "attackStepDescription": attack_descs,
        "attackStepTechnique": attack_techs,
        "cwe_id": cwe_ids,
    }
    items.append(item)

# Write JSONL
with JSONL_OUT.open("w", encoding="utf-8") as fh:
    for it in items:
        fh.write(json.dumps(it, ensure_ascii=False) + "\\n")

# Write flattened CSV (join lists with delimiters)
with CSV_OUT.open("w", encoding="utf-8", newline="") as fh:
    fieldnames = [
        "capec_id", "description",
        "skilllevel", "skilldescription",
        "consequenceImpact", "consequenceScope", "consequenceNote",
        "attackStep", "attackStepPhase", "attackStepDescription", "attackStepTechnique",
        "cwe_id"
    ]
    w = csv.DictWriter(fh, fieldnames=fieldnames)
    w.writeheader()
    for it in items:
        row = {
            "capec_id": it["capec_id"],
            "description": it["description"],
            "skilllevel": ";".join(it["skilllevel"]),
            "skilldescription": ";".join(it["skilldescription"]),
            "consequenceImpact": ";".join(["|".join(x) for x in it["consequenceImpact"]]),
            "consequenceScope": ";".join(["|".join(x) for x in it["consequenceScope"]]),
            "consequenceNote": ";".join(["|".join(x) for x in it["consequenceNote"]]),
            "attackStep": ";".join(it["attackStep"]),
            "attackStepPhase": ";".join(it["attackStepPhase"]),
            "attackStepDescription": ";".join(it["attackStepDescription"]),
            "attackStepTechnique": ";".join(["|".join(x) for x in it["attackStepTechnique"]]),
            "cwe_id": ";".join(it["cwe_id"]),
        }
        w.writerow(row)



